# Proyecto Final AML  
## Predicción de Rotación de Empleados en una Empresa Tech usando Big Data Sintético

**Autor:** Diego Alcázar  
**Curso:** Advanced Machine Learning  

Este proyecto simula un dataset de RRHH a gran escala y construye un modelo de series temporales para predecir la rotación mensual de empleados en una empresa tecnológica.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV

In [ ]:
np.random.seed(42)

n_employees = 20000
n_months = 60

dates = pd.date_range(start="2019-01-01", periods=n_months, freq="M")
employee_ids = np.arange(n_employees)

records = []

for date in dates:
    month_factor = 1 + 0.2*np.sin(date.month/12 * 2*np.pi)

    satisfaction = np.random.beta(2, 2, n_employees).astype("float32")
    workload = np.random.normal(0.6, 0.15, n_employees).clip(0,1).astype("float32")
    absences = np.random.poisson(1.5, n_employees).astype("int8")
    tenure = np.random.randint(1, 120, n_employees).astype("int16")

    prob_leave = (
        0.15*(1-satisfaction)
        + 0.1*workload
        + 0.05*(absences>2)
        + 0.02*(tenure<12)
    ) * month_factor

    leave = np.random.binomial(1, prob_leave.clip(0,0.8)).astype("int8")

    df_temp = pd.DataFrame({
        "employee_id": employee_ids,
        "date": date,
        "satisfaction": satisfaction,
        "workload": workload,
        "absences": absences,
        "tenure": tenure,
        "leave": leave
    })

    records.append(df_temp)

df = pd.concat(records, ignore_index=True)

print("Tamaño del dataset:", df.shape)
df.head()

In [ ]:
monthly = (
    df.groupby("date")
      .agg(
          employees=("employee_id","count"),
          leavers=("leave","sum"),
          avg_satisfaction=("satisfaction","mean"),
          avg_workload=("workload","mean"),
          avg_absences=("absences","mean"),
      )
)

monthly["attrition_rate"] = monthly["leavers"] / monthly["employees"]
monthly.head()

In [ ]:
plt.figure()
plt.plot(monthly.index, monthly["attrition_rate"])
plt.title("Tasa de Rotación en el Tiempo")
plt.xlabel("Fecha")
plt.ylabel("Rotación")
plt.show()

In [ ]:
for lag in [1,2,3]:
    monthly[f"lag_{lag}"] = monthly["attrition_rate"].shift(lag)

monthly["rolling_mean_3"] = monthly["attrition_rate"].rolling(3).mean()

monthly = monthly.dropna()
monthly.head()

In [ ]:
train = monthly.iloc[:-12]
test = monthly.iloc[-12:]

X_train = train.drop(columns=["attrition_rate","employees","leavers"])
y_train = train["attrition_rate"]

X_test = test.drop(columns=["attrition_rate","employees","leavers"])
y_test = test["attrition_rate"]

In [ ]:
baseline = GradientBoostingRegressor(random_state=42)
baseline.fit(X_train, y_train)

preds_base = baseline.predict(X_test)

mae = mean_absolute_error(y_test, preds_base)
mse = mean_squared_error(y_test, preds_base)
rmse = np.sqrt(mse)

print("Baseline MAE:", mae)
print("Baseline RMSE:", rmse)

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2,3]
}

grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

model = grid.best_estimator_
print("Mejores parámetros:", grid.best_params_)

In [ ]:
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)

print("MAE Final:", mae)
print("RMSE Final:", rmse)

In [ ]:
plt.figure()
plt.plot(test.index, y_test.values, label="Real")
plt.plot(test.index, preds, label="Predicción")
plt.legend()
plt.title("Predicción vs Real de Rotación")
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
importances.sort_values().plot(kind="barh")
plt.title("Importancia de Variables")
plt.show()

## Conclusiones

El modelo logra predecir la tendencia de rotación mensual utilizando indicadores agregados de RRHH.

Hallazgos principales:

- La satisfacción laboral y la carga de trabajo influyen significativamente en la rotación.
- Las variables temporales mejoran la capacidad predictiva.
- Este tipo de modelo permite a RRHH planificar contrataciones, presupuestos y programas de retención.

El proyecto demuestra cómo el Machine Learning puede apoyar la planificación estratégica de la fuerza laboral.